# Salary Forecasting With Lagged Contract And Performance Features

This notebook isolates salary modeling from the main data processing notebook. The target is next-season salary cap share, and every modeling feature is known before the target season begins.

Core design:

- Target season `t`: `salary_cap_share_t`.
- Contract feature: previous season salary, `salary_cap_share_t_minus_1`.
- Player context: age and position at target season.
- Performance features: minutes and production from season `t-1` only.

This avoids using same-season performance to predict same-season salary.

In [ ]:
%pip install -q mlflow lightgbm

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None


## Paths And Input Data

The notebook reads the clean gold tables created by the main processing notebook. This keeps cleaning behavior consistent and limits this notebook to salary-specific target construction and modeling.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
COLAB_DATA_DIR = Path("/content/drive/MyDrive/nba-scout-assistant/data")
LOCAL_DATA_DIR = PROJECT_ROOT / "data"


def first_existing_path(paths: list[Path]) -> Path:
    """Input: candidate data roots. Output: first existing path, otherwise local data path."""
    for path in paths:
        if path.exists():
            return path
    return LOCAL_DATA_DIR


DATA_DIR = Path(os.getenv("NBA_SCOUT_DATA_DIR", "")).expanduser() if os.getenv("NBA_SCOUT_DATA_DIR") else first_existing_path([COLAB_DATA_DIR, LOCAL_DATA_DIR])
GOLD_DIR = DATA_DIR / "gold"
MLFLOW_BACKEND_DB = DATA_DIR.parent / "mlflow.db"
MLFLOW_ARTIFACT_DIR = DATA_DIR.parent / "mlartifacts"
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SALARY_PATH = GOLD_DIR / "salary_training_clean.parquet"
ROLE_PATH = GOLD_DIR / "player_role_features_clean.parquet"

print("DATA_DIR:", DATA_DIR)
print("SALARY_PATH exists:", SALARY_PATH.exists())
print("ROLE_PATH exists:", ROLE_PATH.exists())

salary_clean = pd.read_parquet(SALARY_PATH)
role_clean = pd.read_parquet(ROLE_PATH)

print("salary_clean", salary_clean.shape)
print("role_clean", role_clean.shape)


## MLflow And Cache

Metrics are cached by a settings signature. Re-running the notebook will load cached metric tables when the input data and experiment settings are unchanged.

In [ ]:
MLFLOW_EXPERIMENT_NAME = "nba_scout_salary_forecasting"
TRAINING_CACHE_ENABLED = os.getenv("NBA_SCOUT_USE_MLFLOW_CACHE", "true").lower() in {"1", "true", "yes"}
METRIC_CACHE_DIR = MLFLOW_ARTIFACT_DIR / "metric_cache"
METRIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def try_import_mlflow():
    """Input: none. Output: mlflow module or None when unavailable."""
    try:
        import mlflow  # type: ignore
        return mlflow
    except ImportError:
        print("MLflow is not installed. Metrics will still be saved to parquet.")
        return None


def configure_mlflow():
    """Input: none. Output: configured mlflow module or None when tracking is unavailable."""
    mlflow = try_import_mlflow()
    if mlflow is None:
        return None
    tracking_uri = f"sqlite:///{MLFLOW_BACKEND_DB}"
    artifact_uri = MLFLOW_ARTIFACT_DIR.as_uri()
    mlflow.set_tracking_uri(tracking_uri)
    if mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME) is None:
        mlflow.create_experiment(MLFLOW_EXPERIMENT_NAME, artifact_location=artifact_uri)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    print("MLflow tracking URI:", tracking_uri)
    print("MLflow artifact URI:", artifact_uri)
    print("MLflow experiment:", MLFLOW_EXPERIMENT_NAME)
    return mlflow


def json_safe(value):
    """Input: arbitrary object. Output: JSON-stable representation for signatures."""
    if isinstance(value, dict):
        return {str(key): json_safe(value[key]) for key in sorted(value)}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def settings_signature(payload: dict[str, object]) -> str:
    """Input: settings payload. Output: stable short hash for cache validation."""
    encoded = json.dumps(json_safe(payload), sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def dataframe_fingerprint(df: pd.DataFrame, split_col: str | None = None) -> dict[str, object]:
    """Input: dataframe. Output: compact fingerprint for cache invalidation."""
    fingerprint: dict[str, object] = {"rows": len(df), "columns": list(df.columns)}
    if split_col and split_col in df.columns:
        fingerprint["split_counts"] = df[split_col].value_counts(dropna=False).sort_index().to_dict()
    for column in ["season", "season_label"]:
        if column in df.columns:
            values = df[column].dropna().astype(str)
            fingerprint[f"{column}_min"] = values.min() if not values.empty else None
            fingerprint[f"{column}_max"] = values.max() if not values.empty else None
            fingerprint[f"{column}_nunique"] = int(values.nunique())
    return fingerprint


def metric_cache_paths(cache_name: str) -> tuple[Path, Path]:
    """Input: cache name. Output: parquet path and JSON signature path."""
    return METRIC_CACHE_DIR / f"{cache_name}.parquet", METRIC_CACHE_DIR / f"{cache_name}.signature.json"


def load_metric_cache(cache_name: str, signature: str) -> pd.DataFrame | None:
    """Input: cache name and expected signature. Output: cached dataframe or None."""
    if not TRAINING_CACHE_ENABLED:
        return None
    data_path, signature_path = metric_cache_paths(cache_name)
    if not data_path.exists() or not signature_path.exists():
        return None
    cached_signature = json.loads(signature_path.read_text()).get("settings_signature")
    if cached_signature != signature:
        print(f"Metric cache miss for {cache_name}: settings changed.")
        return None
    cached = pd.read_parquet(data_path)
    print(f"Loaded cached metrics for {cache_name}: {data_path} shape={cached.shape}")
    return cached


def save_metric_cache(cache_name: str, signature: str, df: pd.DataFrame, payload: dict[str, object]) -> None:
    """Input: cache metadata and dataframe. Output: persisted metric cache."""
    if df.empty:
        return
    data_path, signature_path = metric_cache_paths(cache_name)
    df.to_parquet(data_path, index=False)
    signature_path.write_text(json.dumps({"settings_signature": signature, "payload": json_safe(payload)}, indent=2, sort_keys=True))
    print(f"Saved metric cache for {cache_name}: {data_path} shape={df.shape}")


mlflow_client = configure_mlflow()


## Build Lagged Salary Dataset

Each row predicts salary in target season `t`. Previous salary and previous performance are joined from season `t-1` by `player_id`.

In [ ]:
SALARY_TARGET = "salary_cap_share"
MODERN_TRAIN_MAX_SEASON_START = 2022
MODERN_VALIDATION_SEASON_START = 2023
MODERN_TEST_SEASON_START = 2024


def assign_lagged_salary_split(season_start_year: object) -> str:
    """Input: target season start year. Output: temporal split for lagged salary forecasting."""
    year = int(season_start_year)
    if year <= MODERN_TRAIN_MAX_SEASON_START:
        return "train"
    if year == MODERN_VALIDATION_SEASON_START:
        return "validation"
    if year == MODERN_TEST_SEASON_START:
        return "test"
    return "ignore"


def build_lagged_salary_dataset(salary_df: pd.DataFrame, role_df: pd.DataFrame) -> pd.DataFrame:
    """Input: clean salary and role tables. Output: target-season salary rows with only prior-season features."""
    salary_cols = [
        "player_id", "player_name", "season_label", "season_start_year", "age", "position",
        "height", "weight", "salary_usd", "salary_cap_usd", "salary_cap_share",
    ]
    salary_base = salary_df[salary_cols].copy()
    salary_base = salary_base.dropna(subset=["player_id", "season_start_year", "salary_usd", "salary_cap_usd", "salary_cap_share"]).copy()
    salary_base["player_id"] = salary_base["player_id"].astype("Int64")
    salary_base["season_start_year"] = salary_base["season_start_year"].astype(int)
    salary_base["previous_season_start_year"] = salary_base["season_start_year"] - 1

    previous_salary = salary_base[["player_id", "season_start_year", "salary_usd", "salary_cap_share"]].rename(columns={
        "season_start_year": "previous_season_start_year",
        "salary_usd": "previous_salary_usd",
        "salary_cap_share": "previous_salary_cap_share",
    })

    previous_role = role_df.copy()
    previous_role["previous_season_start_year"] = previous_role["season"].astype(str).str.slice(0, 4).astype(int)
    previous_role = previous_role.rename(columns={
        column: f"prev_{column}"
        for column in previous_role.columns
        if column not in {"player_id", "previous_season_start_year"}
    })

    result = salary_base.merge(previous_salary, on=["player_id", "previous_season_start_year"], how="inner")
    result = result.merge(previous_role, on=["player_id", "previous_season_start_year"], how="inner")
    result["salary_delta_cap_share"] = result["salary_cap_share"] - result["previous_salary_cap_share"]
    result["salary_ratio_to_previous"] = result["salary_cap_share"] / result["previous_salary_cap_share"].replace(0, np.nan)
    result["split"] = result["season_start_year"].map(assign_lagged_salary_split)
    result = result[result["split"].isin(["train", "validation", "test"])].copy()
    return result.reset_index(drop=True)


lagged_salary = build_lagged_salary_dataset(salary_clean, role_clean)

print("lagged_salary", lagged_salary.shape)
display(lagged_salary.groupby(["season_label", "split"]).size().reset_index(name="rows"))
display(lagged_salary.head())


## Feature Sets

The feature sets start from the expected drivers: previous salary, age, previous minutes, and prior-season performance. Baselines are included because previous salary cap share is a very strong carry-forward benchmark.

In [ ]:
LAGGED_SALARY_FEATURE_SETS = {
    "contract_age_prev_minutes": [
        "previous_salary_cap_share", "previous_salary_usd", "age", "prev_minutes", "position",
    ],
    "contract_age_role_core": [
        "previous_salary_cap_share", "previous_salary_usd", "age", "prev_minutes", "prev_usage_pct",
        "prev_scoring_creation", "prev_playmaking", "prev_rebounding", "position",
    ],
    "contract_age_selected_perf": [
        "previous_salary_cap_share", "previous_salary_usd", "age", "prev_minutes", "prev_usage_pct",
        "prev_points_per_100", "prev_assists_per_100", "prev_rebounds_per_100", "prev_true_shooting_pct", "position",
    ],
    "contract_age_full_role": [
        "previous_salary_cap_share", "previous_salary_usd", "age", "height", "weight", "prev_minutes",
        "prev_usage_pct", "prev_scoring_creation", "prev_playmaking", "prev_shooting", "prev_rim_pressure",
        "prev_rebounding", "prev_perimeter_defense", "prev_interior_defense", "prev_two_way_impact", "position",
    ],
}

LAGGED_SALARY_BASELINES = ["previous_salary_cap_share", "train_median"]
display(pd.DataFrame([
    {"feature_set": name, "feature_count": len([feature for feature in features if feature in lagged_salary.columns]), "features": ", ".join([feature for feature in features if feature in lagged_salary.columns])}
    for name, features in LAGGED_SALARY_FEATURE_SETS.items()
]))


## Train And Evaluate

Primary metric is cap-share MAE. USD MAE is reported by converting predicted cap share back to salary dollars using the target season cap.

In [ ]:
LAGGED_SALARY_EVALUATION_PATH = GOLD_DIR / "lagged_salary_model_evaluation.parquet"
LAGGED_SALARY_SELECTION_PATH = GOLD_DIR / "lagged_salary_model_selection_summary.parquet"
LAGGED_SALARY_FEATURE_IMPORTANCE_PATH = GOLD_DIR / "lagged_salary_feature_importance.parquet"


def get_one_hot_encoder():
    """Input: none. Output: OneHotEncoder compatible with the local sklearn version."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(df: pd.DataFrame, features: list[str], scale_numeric: bool = False) -> ColumnTransformer:
    """Input: dataframe, feature names, scale flag. Output: sklearn preprocessor."""
    numeric_features = [feature for feature in features if pd.api.types.is_numeric_dtype(df[feature])]
    categorical_features = [feature for feature in features if feature not in numeric_features]
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric_features),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", get_one_hot_encoder())]), categorical_features),
    ])


def build_estimators() -> dict[str, tuple[object, bool]]:
    """Input: none. Output: candidate salary estimators and whether numeric scaling is needed."""
    estimators: dict[str, tuple[object, bool]] = {
        "ridge": (Ridge(alpha=5.0), True),
        "elasticnet": (ElasticNet(alpha=0.001, l1_ratio=0.2, max_iter=20000, random_state=42), True),
        "random_forest": (RandomForestRegressor(n_estimators=400, min_samples_leaf=8, max_features="sqrt", n_jobs=-1, random_state=42), False),
        "hist_gradient_boosting": (HistGradientBoostingRegressor(max_iter=300, learning_rate=0.035, l2_regularization=0.05, min_samples_leaf=20, random_state=42), False),
        "mlp": (MLPRegressor(hidden_layer_sizes=(64, 32), alpha=0.001, learning_rate_init=0.001, early_stopping=True, validation_fraction=0.15, max_iter=500, random_state=42), True),
    }
    if LGBMRegressor is not None:
        estimators["lightgbm"] = (LGBMRegressor(n_estimators=500, learning_rate=0.025, num_leaves=31, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, random_state=42, verbosity=-1), False)
    return estimators


def evaluate_predictions(df: pd.DataFrame, split: str, pred_share: np.ndarray) -> dict[str, object]:
    """Input: split dataframe and cap-share predictions. Output: salary regression metrics."""
    y_true = df[SALARY_TARGET].astype(float)
    pred_share = np.clip(np.asarray(pred_share, dtype="float64"), 0, None)
    pred_usd = pred_share * df["salary_cap_usd"].astype(float)
    return {
        "split": split,
        "rows": len(df),
        "cap_share_mae": mean_absolute_error(y_true, pred_share),
        "usd_mae": mean_absolute_error(df["salary_usd"].astype(float), pred_usd),
        "rmse": mean_squared_error(y_true, pred_share) ** 0.5,
        "r2": r2_score(y_true, pred_share),
    }


def log_metrics_to_mlflow(mlflow, evaluation_df: pd.DataFrame, cache_signature: str | None = None) -> None:
    """Input: mlflow module and evaluation dataframe. Output: logged metric run."""
    if mlflow is None or evaluation_df.empty:
        return
    with mlflow.start_run(run_name="lagged_salary_forecasting"):
        mlflow.set_tags({"project": "nba-scout-assistant", "experiment_type": "lagged_salary_forecasting", "target": SALARY_TARGET})
        if cache_signature is not None:
            mlflow.log_param("cache_signature", cache_signature)
        for row in evaluation_df.itertuples(index=False):
            prefix = f"{row.split}_{row.model}_{row.feature_set}"
            mlflow.log_metric(f"{prefix}_cap_share_mae", float(row.cap_share_mae))
            mlflow.log_metric(f"{prefix}_usd_mae", float(row.usd_mae))
            mlflow.log_metric(f"{prefix}_r2", float(row.r2))


def train_lagged_salary_models(df: pd.DataFrame) -> tuple[dict[str, Pipeline], pd.DataFrame, pd.DataFrame]:
    """Input: lagged salary data. Output: fitted models, evaluation table, and selected summary."""
    train_df = df[df["split"].eq("train")].copy()
    rows = []
    models: dict[str, Pipeline] = {}

    train_median = float(train_df[SALARY_TARGET].median())
    for split_name in ["validation", "test"]:
        split_df = df[df["split"].eq(split_name)].copy()
        baseline_predictions = {
            "previous_salary_cap_share": split_df["previous_salary_cap_share"].to_numpy(dtype="float64"),
            "train_median": np.full(len(split_df), train_median),
        }
        for baseline_name, pred_share in baseline_predictions.items():
            metrics = evaluate_predictions(split_df, split_name, pred_share)
            rows.append({"experiment": "lagged_salary", "model": baseline_name, "feature_set": "baseline", "features": baseline_name, **metrics})

    estimators = build_estimators()
    for feature_set_name, raw_features in LAGGED_SALARY_FEATURE_SETS.items():
        features = [feature for feature in raw_features if feature in df.columns]
        for model_name, (estimator, scale_numeric) in estimators.items():
            model = Pipeline([
                ("preprocess", build_preprocessor(df, features, scale_numeric=scale_numeric)),
                ("model", clone(estimator)),
            ])
            model.fit(train_df[features], train_df[SALARY_TARGET].astype(float))
            model_key = f"{model_name}_{feature_set_name}"
            models[model_key] = model
            for split_name in ["validation", "test"]:
                split_df = df[df["split"].eq(split_name)].copy()
                pred_share = model.predict(split_df[features])
                metrics = evaluate_predictions(split_df, split_name, pred_share)
                rows.append({"experiment": "lagged_salary", "model": model_name, "feature_set": feature_set_name, "features": ", ".join(features), **metrics})

    evaluation = pd.DataFrame(rows).sort_values(["split", "cap_share_mae"]).reset_index(drop=True)
    validation = evaluation[evaluation["split"].eq("validation")].copy()
    best = validation.sort_values("cap_share_mae").iloc[0]
    test_row = evaluation[
        evaluation["split"].eq("test")
        & evaluation["model"].eq(best["model"])
        & evaluation["feature_set"].eq(best["feature_set"])
    ].head(1)
    selection = pd.DataFrame([{
        "target": SALARY_TARGET,
        "selected_by": "validation_cap_share_mae",
        "recommended_model": best["model"],
        "feature_set": best["feature_set"],
        "validation_cap_share_mae": best["cap_share_mae"],
        "validation_usd_mae": best["usd_mae"],
        "validation_r2": best["r2"],
        "test_cap_share_mae": test_row.iloc[0]["cap_share_mae"] if not test_row.empty else np.nan,
        "test_usd_mae": test_row.iloc[0]["usd_mae"] if not test_row.empty else np.nan,
        "test_r2": test_row.iloc[0]["r2"] if not test_row.empty else np.nan,
    }])
    return models, evaluation, selection


lagged_salary_cache_payload = {
    "cache_version": "lagged_salary_forecasting_v1",
    "feature_sets": LAGGED_SALARY_FEATURE_SETS,
    "target": SALARY_TARGET,
    "data": dataframe_fingerprint(lagged_salary, split_col="split"),
}
lagged_salary_cache_signature = settings_signature(lagged_salary_cache_payload)
lagged_salary_evaluation = load_metric_cache("lagged_salary_model_evaluation", lagged_salary_cache_signature)
lagged_salary_selection = load_metric_cache("lagged_salary_model_selection", lagged_salary_cache_signature)

if lagged_salary_evaluation is None or lagged_salary_selection is None:
    lagged_salary_models, lagged_salary_evaluation, lagged_salary_selection = train_lagged_salary_models(lagged_salary)
    save_metric_cache("lagged_salary_model_evaluation", lagged_salary_cache_signature, lagged_salary_evaluation, lagged_salary_cache_payload)
    save_metric_cache("lagged_salary_model_selection", lagged_salary_cache_signature, lagged_salary_selection, lagged_salary_cache_payload)
    log_metrics_to_mlflow(mlflow_client, lagged_salary_evaluation, lagged_salary_cache_signature)
else:
    lagged_salary_models = {}

lagged_salary_evaluation.to_parquet(LAGGED_SALARY_EVALUATION_PATH, index=False)
lagged_salary_selection.to_parquet(LAGGED_SALARY_SELECTION_PATH, index=False)

display(lagged_salary_selection)
display(lagged_salary_evaluation)


## Feature Analysis

This section evaluates which prior-season performance features are most related to next-season salary cap share. It reports linear correlation, mutual information, and permutation importance for the selected non-baseline model when available.

In [ ]:
def analyze_lagged_salary_features(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Input: lagged salary data and features. Output: correlation and mutual-information analysis."""
    train_df = df[df["split"].eq("train")].copy()
    numeric_features = [feature for feature in features if feature in train_df.columns and pd.api.types.is_numeric_dtype(train_df[feature])]
    rows = []
    for feature in numeric_features:
        pair = pd.concat([pd.to_numeric(train_df[feature], errors="coerce"), train_df[SALARY_TARGET]], axis=1).dropna()
        if len(pair) < 50 or pair.iloc[:, 0].nunique() <= 1:
            continue
        corr = pair.iloc[:, 0].corr(pair.iloc[:, 1])
        rows.append({"analysis": "correlation", "feature": feature, "score": corr, "abs_score": abs(corr)})

    X_numeric = train_df[numeric_features].copy()
    X_numeric = X_numeric.fillna(X_numeric.median(numeric_only=True))
    mi_scores = mutual_info_regression(X_numeric, train_df[SALARY_TARGET].astype(float), random_state=42)
    rows.extend([
        {"analysis": "mutual_info", "feature": feature, "score": score, "abs_score": score}
        for feature, score in zip(numeric_features, mi_scores)
    ])
    return pd.DataFrame(rows).sort_values(["analysis", "abs_score"], ascending=[True, False]).reset_index(drop=True)


def permutation_importance_for_best_model(df: pd.DataFrame, models: dict[str, Pipeline], evaluation_df: pd.DataFrame) -> pd.DataFrame:
    """Input: lagged salary data, fitted models, evaluation. Output: validation permutation importance."""
    validation_models = evaluation_df[
        evaluation_df["split"].eq("validation")
        & ~evaluation_df["feature_set"].eq("baseline")
    ].copy()
    if validation_models.empty:
        return pd.DataFrame()
    best = validation_models.sort_values("cap_share_mae").iloc[0]
    model_key = f"{best['model']}_{best['feature_set']}"
    if model_key not in models:
        return pd.DataFrame()
    features = [feature for feature in LAGGED_SALARY_FEATURE_SETS[str(best["feature_set"])] if feature in df.columns]
    validation_df = df[df["split"].eq("validation")].copy()
    result = permutation_importance(
        models[model_key],
        validation_df[features],
        validation_df[SALARY_TARGET].astype(float),
        scoring="neg_mean_absolute_error",
        n_repeats=5,
        random_state=42,
    )
    return pd.DataFrame({
        "analysis": "permutation_mae_increase",
        "feature": features,
        "score": result.importances_mean,
        "abs_score": np.abs(result.importances_mean),
        "importance_std": result.importances_std,
    }).sort_values("abs_score", ascending=False).reset_index(drop=True)


analysis_features = sorted({feature for features in LAGGED_SALARY_FEATURE_SETS.values() for feature in features if feature in lagged_salary.columns})
lagged_salary_feature_analysis = analyze_lagged_salary_features(lagged_salary, analysis_features)
permutation_analysis = permutation_importance_for_best_model(lagged_salary, lagged_salary_models, lagged_salary_evaluation)
if not permutation_analysis.empty:
    lagged_salary_feature_analysis = pd.concat([lagged_salary_feature_analysis, permutation_analysis], ignore_index=True)

lagged_salary_feature_analysis.to_parquet(LAGGED_SALARY_FEATURE_IMPORTANCE_PATH, index=False)
display(lagged_salary_feature_analysis)


## Initial Reading Guide

If `previous_salary_cap_share` wins, the salary market is mostly behaving like contract carry-forward in this dataset. If a model with prior performance beats it, inspect the feature analysis to see which prior performance signals improve next-season salary prediction.